# PRG4206 — Google Colab Run and Outputs
**Student:** Vishnu Vardhan · **Student ID:** I26043592  
**Project:** A Comparative and Explainable Machine Learning System for AI-Generated Text Detection

This notebook checks the saved project in a fresh Google Colab runtime. It installs the lightweight, version-aligned dependencies; runs the automated test suite; independently verifies the saved test predictions and statistical comparisons; executes text predictions with all three saved classifiers; tests TXT/PDF/DOCX document parsing; and creates `outputs/PRG4206_Project_Outputs_Summary.pdf` plus a new ZIP containing the runtime report.

**Important scope:** this is a reproducibility/verification run of the three bundled models, not a fresh re-training of the models and not a DistilBERT benchmark. The report is generated from the results and logs actually observed when you execute the cells. Do not present an unexecuted or failed run as a successful Colab run.

## 1. Upload the code package
Upload `I26043592_Vishnu_Vardhan_Code_ColabReady.zip` (or the latest code ZIP you prepared) when prompted. The archive should contain this notebook, `requirements-colab.txt`, `scripts/generate_output_report.py`, the saved models, test predictions, metrics and sample documents.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile, os, sys, subprocess, json, platform, datetime, textwrap

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f"Upload exactly one project ZIP file; found {zip_names}")
ZIP_PATH = Path('/content') / zip_names[0]
EXTRACT_ROOT = Path('/content/prg4206_project_extract')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

# Safe extraction: reject paths that try to escape the extraction directory.
with zipfile.ZipFile(ZIP_PATH) as zf:
    for member in zf.infolist():
        destination = (EXTRACT_ROOT / member.filename).resolve()
        if EXTRACT_ROOT.resolve() not in destination.parents and destination != EXTRACT_ROOT.resolve():
            raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")
    zf.extractall(EXTRACT_ROOT)

candidates = [p for p in EXTRACT_ROOT.rglob('README.md') if (p.parent / 'src').is_dir() and (p.parent / 'models').is_dir()]
if not candidates:
    raise FileNotFoundError('Could not find the project root (README.md, src/, and models/). Please upload the project ZIP.')
PROJECT_ROOT = candidates[0].parent
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Uploaded archive:', ZIP_PATH.name, f'({ZIP_PATH.stat().st_size:,} bytes)')
print('Required files found:', all((PROJECT_ROOT / p).exists() for p in ['requirements-colab.txt','src/verify_statistics.py','models/svm/svm_pipeline.joblib','models/logistic_regression/lr_pipeline.joblib','models/xgboost/xgb_model.joblib']))

## 2. Install the project verification dependencies
The notebook uses the dedicated `requirements-colab.txt` to align the main package versions with the saved model artefacts. This avoids installing the optional transformer-training stack, which is not needed for the saved-model verification run.

In [ ]:
install_cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-colab.txt')]
install = subprocess.run(install_cmd, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(install.stdout[-8000:])
if install.returncode != 0:
    raise RuntimeError('Dependency installation failed. Read the output above, fix the dependency issue, and rerun this cell.')
print('Dependency installation completed.')

## 3. Run the automated tests and independent statistical verification
These commands test the packaged project and recalculate metrics/statistical comparisons from the saved per-example predictions. The outputs below are from this live runtime.

In [ ]:
from datetime import datetime, timezone

def run_command(command, timeout=1200):
    result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    return {
        'command': ' '.join(str(x) for x in command),
        'returncode': int(result.returncode),
        'stdout': result.stdout,
        'stderr': '',
        'summary': next((line.strip() for line in reversed(result.stdout.splitlines()) if line.strip()), f'exit code {result.returncode}')
    }

test_result = run_command([sys.executable, '-m', 'pytest', 'tests/', '-q'], timeout=1200)
print('COMMAND:', test_result['command'])
print(test_result['stdout'][-12000:])
print('TEST EXIT CODE:', test_result['returncode'])

statistics_result = run_command([sys.executable, '-m', 'src.verify_statistics'], timeout=1200)
print('COMMAND:', statistics_result['command'])
print(statistics_result['stdout'][-12000:])
print('STATISTICAL VERIFICATION EXIT CODE:', statistics_result['returncode'])

## 4. Run live predictions with all three saved models
Two included sample passages are run through Logistic Regression, Linear SVM and XGBoost. These are inference demonstrations, not new benchmark accuracy measurements; report each model's output exactly as it appears.

In [ ]:
from src.inference import predict_text
from src.document_parser import extract_text_from_file, analyze_document

sample_dir = PROJECT_ROOT / 'sample_documents'
human_path = sample_dir / 'human_sample_essay.txt'
ai_docx_path = sample_dir / 'ai_sample_overview.docx'
if not human_path.exists() or not ai_docx_path.exists():
    raise FileNotFoundError('Required sample demonstration documents were not found in sample_documents/.')

human_text = human_path.read_text(encoding='utf-8', errors='replace').strip()
ai_doc_data = extract_text_from_file(ai_docx_path, filename=ai_docx_path.name, min_text_words=10)
ai_text = ai_doc_data['full_text'].strip()
demo_cases = {
    'sample text document': human_text[:4000],
    'sample structured document': ai_text[:4000],
}
prediction_rows = []
inference_errors = []
for case_name, sample in demo_cases.items():
    for model_name in ['lr', 'svm', 'xgb']:
        try:
            result = predict_text(sample, model_name=model_name)
            prediction_rows.append({
                'case': case_name,
                'model': result.get('model', model_name),
                'label': result['label'],
                'confidence': float(result['confidence']),
                'word_count': int(result['text_stats']['word_count']),
                'char_count': int(result['text_stats']['char_count']),
            })
        except Exception as exc:
            inference_errors.append({'case':case_name,'model':model_name,'error':repr(exc)})

import pandas as pd
predictions_df = pd.DataFrame(prediction_rows)
display(predictions_df)
if inference_errors:
    print('INFERENCE ERRORS:', inference_errors)
all_models_succeeded = len(prediction_rows) == 6 and not inference_errors
print('All three saved models produced predictions for both examples:', all_models_succeeded)

## 5. Check TXT, PDF and DOCX document parsing
Each sample document is extracted and passed through the document-analysis pipeline using the saved Linear SVM. The resulting document label is an aggregate of chunk predictions, not a separately benchmarked document-level accuracy result.

In [ ]:
document_checks = []
for path in sorted(sample_dir.iterdir()):
    if path.suffix.lower() not in {'.txt', '.pdf', '.docx'}:
        continue
    item = {'filename': path.name, 'format': path.suffix.lower(), 'success': False}
    try:
        doc = extract_text_from_file(path, filename=path.name, min_text_words=10)
        analysis = analyze_document(doc, model_name='svm', min_chunk_words=20, max_chunk_words=150)
        item.update({
            'success': True,
            'word_count': int(doc.get('word_count', len(doc.get('full_text','').split()))),
            'chunks': int(analysis['total_chunks']),
            'prediction': str(analysis['overall_prediction']),
            'confidence': float(analysis['overall_confidence']),
            'error': ''
        })
    except Exception as exc:
        item['error'] = repr(exc)
    document_checks.append(item)

display(pd.DataFrame(document_checks))
document_checks_all_succeeded = bool(document_checks) and all(x['success'] for x in document_checks)
print('All available TXT/PDF/DOCX sample checks succeeded:', document_checks_all_succeeded)

## 6. Save the actual Colab runtime record and generate the outputs PDF
The runtime record includes timestamp, Python/platform information, test output, statistical verification output, live model predictions and document-parser results. The PDF status is calculated from these checks rather than hard-coded as a pass.

In [ ]:
outputs_dir = PROJECT_ROOT / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
runtime_record = {
    'runtime_kind': 'Google Colab',
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'timestamp_local': datetime.now().astimezone().isoformat(),
    'project_root': str(PROJECT_ROOT),
    'environment': {'python': sys.version, 'platform': platform.platform()},
    'tests': test_result,
    'statistics': statistics_result,
    'inference': {
        'all_models_succeeded': bool(all_models_succeeded),
        'predictions': prediction_rows,
        'errors': inference_errors,
    },
    'document_checks': {
        'all_succeeded': bool(document_checks_all_succeeded),
        'items': document_checks,
    }
}
runtime_json_path = outputs_dir / 'colab_runtime.json'
runtime_json_path.write_text(json.dumps(runtime_record, indent=2), encoding='utf-8')

pdf_path = outputs_dir / 'PRG4206_Project_Outputs_Summary.pdf'
pdf_cmd = [sys.executable, 'scripts/generate_output_report.py', '--project-root', str(PROJECT_ROOT), '--output', str(pdf_path), '--runtime-json', str(runtime_json_path)]
pdf_result = subprocess.run(pdf_cmd, cwd=PROJECT_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(pdf_result.stdout)
if pdf_result.returncode != 0 or not pdf_path.exists():
    raise RuntimeError('The outputs PDF could not be generated. Review the error above.')
print('Outputs PDF:', pdf_path, f'({pdf_path.stat().st_size:,} bytes)')

## 7. Create a new ZIP with the Colab outputs attached
This final archive contains the project files, the notebook, the actual runtime JSON, and the PDF generated above. The original ZIP uploaded at the beginning is not overwritten.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

final_zip = Path('/content/I26043592_Vishnu_Vardhan_Code.zip')
if final_zip.exists():
    final_zip.unlink()
excluded_parts = {'.git', '.venv', '__pycache__', '.pytest_cache', '.ipynb_checkpoints'}
with ZipFile(final_zip, 'w', compression=ZIP_DEFLATED, compresslevel=6) as zf:
    for path in PROJECT_ROOT.rglob('*'):
        if not path.is_file() or any(part in excluded_parts for part in path.parts):
            continue
        arcname = Path(PROJECT_ROOT.name) / path.relative_to(PROJECT_ROOT)
        zf.write(path, arcname.as_posix())

print('Final code/output ZIP:', final_zip, f'({final_zip.stat().st_size:,} bytes)')
print('Output PDF:', pdf_path)
print('Tests passed exit status:', test_result['returncode'] == 0)
print('Statistics verified exit status:', statistics_result['returncode'] == 0)
print('All model inference succeeded:', all_models_succeeded)
print('Document format checks succeeded:', document_checks_all_succeeded)
print('IMPORTANT: If any check above is False, inspect the logs and do not describe the run as fully passing.')

In [ ]:
# Download the PDF and the updated ZIP to your computer.
from google.colab import files
files.download(str(pdf_path))
files.download(str(final_zip))

## What to submit after you run this notebook
- `outputs/PRG4206_Project_Outputs_Summary.pdf` generated by this executed Colab session.
- `I26043592_Vishnu_Vardhan_Code.zip` downloaded from the final cell.
- The GitHub repository URL, if your lecturer asks for a code link. Do not paste a placeholder URL; use the real repository link after the repository has been created and checked.

The result PDF will state when tests or checks fail; fix those issues and rerun the relevant cells before reporting a successful run.